# HLLSet DSL -- Client Demo

External Python client exercising the full HLLSet pipeline.
**Run cells top-to-bottom.** Kernel: Python 3.

---
## Setup
Import helpers and verify hllset CLI works.

In [1]:
import sys, os, json, subprocess
from pathlib import Path

def find_hllset():
    if env := os.environ.get("HLLSET_BINARY"):
        return env
    for parent in [Path.cwd()] + [p for p in Path.cwd().parents]:
        for name in ["target/release/hllset", "target/debug/hllset"]:
            p = parent / name
            if p.exists():
                return str(p)
    return "hllset"

HLLSET = find_hllset()

def hllset(script):
    proc = subprocess.run(
        [HLLSET, "-e", script],
        capture_output=True, text=True, timeout=10
    )
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip())
    return json.loads(proc.stdout.strip())

def lua_list(items):
    """Convert Python list to Lua table: ['a','b'] -> {'a','b'}."""
    return "{" + ", ".join(json.dumps(i) for i in items) + "}"

def inscribe(tokens):
    j = lua_list(tokens)
    return hllset(f"return hllset.inscribe({j}):key()")

def tokenize(text):
    escaped = text.replace('"', '\\"')
    return hllset(f'return hllset.tokenize("{escaped}"):key()')

def cardinality(text):
    escaped = text.replace('"', '\\"')
    return hllset(f'return #hllset.tokenize("{escaped}")')

def bss_inclusion(text_a, text_b):
    ta = text_a.replace('"', '\\"')
    tb = text_b.replace('"', '\\"')
    return hllset(f'local a=hllset.tokenize("{ta}"); local b=hllset.tokenize("{tb}"); return a:bss_inclusion(b)')

def jaccard(text_a, text_b):
    ta = text_a.replace('"', '\\"')
    tb = text_b.replace('"', '\\"')
    return hllset(f'local a=hllset.tokenize("{ta}"); local b=hllset.tokenize("{tb}"); return a:jaccard(b)')

print(f"hllset: {HLLSET}")
print(f"test: {hllset('return \"ok\"')}")


hllset: /home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/target/release/hllset
test: ok


---
## 1. Text Ingestion
Convert text to content-addressed HLLSet fingerprints.

In [2]:
texts = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "the cat and the dog played together",
]

for i, text in enumerate(texts):
    key = tokenize(text)
    card = cardinality(text)
    print(f"Text {i+1}: key={key}, cardinality={card:.0f}")
    print(f'       "{text}"')
    print()


Text 1: key=h:585f089a1aac49bfd8119bc8a7661b954ccb5df0, cardinality=5
       "the cat sat on the mat"

Text 2: key=h:c804a3b1a6a159b527081ef0848f44f43f35255a, cardinality=5
       "the dog ran in the park"

Text 3: key=h:cb7cc511e3e9da141ef5e7afd7e087dcc6fee20d, cardinality=6
       "the cat and the dog played together"



---
## 2. Document Comparison
O(1) BSS and Jaccard on 1024-bit HLLSets.

In [3]:
pairs = [
    ("machine learning is transforming software engineering",
     "deep learning and machine learning are related fields"),
    ("machine learning is transforming software engineering",
     "quantum physics explores subatomic particles"),
]

for a, b in pairs:
    tau = bss_inclusion(a, b)
    jac = jaccard(a, b)
    print(f"A: {a[:55]}...")
    print(f"B: {b[:55]}...")
    print(f"  BSS tau (A->B): {tau:.3f}")
    print(f"  Jaccard:         {jac:.3f}")
    print()


A: machine learning is transforming software engineering...
B: deep learning and machine learning are related fields...
  BSS tau (A->B): 0.286
  Jaccard:         0.182

A: machine learning is transforming software engineering...
B: quantum physics explores subatomic particles...
  BSS tau (A->B): 0.000
  Jaccard:         0.000



---
## 3. Lattice Operations
Union (+) and intersection (*) via Lua.

In [4]:
a_tokens = ["lattice", "algebra", "bss", "morphism"]
b_tokens = ["algebra", "crdt", "distributed", "bss"]

ka = inscribe(a_tokens)
kb = inscribe(b_tokens)

la = lua_list(a_tokens)
lb = lua_list(b_tokens)

union_card = hllset(
    f"local a=hllset.inscribe({la});"
    f"local b=hllset.inscribe({lb});"
    f"return #(a+b)"
)

inter_card = hllset(
    f"local a=hllset.inscribe({la});"
    f"local b=hllset.inscribe({lb});"
    f"return #(a*b)"
)

print(f"Set A: {a_tokens}")
print(f"Set B: {b_tokens}")
print(f"  Union (A U B):         {union_card}")
print(f"  Intersection (A n B):  {inter_card}")


Set A: ['lattice', 'algebra', 'bss', 'morphism']
Set B: ['algebra', 'crdt', 'distributed', 'bss']
  Union (A U B):         6.0
  Intersection (A n B):  2.0


---
## 4. Materialization Roundtrip
Text -> HLLSet -> tokens.

In [5]:
text = "the quick brown fox jumps over the lazy dog"
tokens = text.lower().split()

key = tokenize(text)
card = cardinality(text)

# Materialize: pass the text directly to Lua tokenize+materialize
lt = lua_list(tokens)
script = (
    "local e = hllset.tokenize('" + text + "'); "
    "return hllset.materialize(e, " + lt + ")"
)
result = hllset(script)

conf = result.get("confidence", 0)
recovered = result.get("tokens", [])
missing = set(tokens) - set(recovered)

print(f"Input:       {text}")
print(f"Key:         {key}")
print(f"Cardinality: {card}")
print(f"Confidence:  {conf:.2%}")
print(f"Recovered:   {len(recovered)} tokens")
if missing:
    print(f"MISSING:     {missing}")
else:
    print(f"FULL ROUNDTRIP -- all {len(set(tokens))} unique tokens recovered")


Input:       the quick brown fox jumps over the lazy dog
Key:         h:f299b3b5c4679f68b3d940f1592a5225c29070d7
Cardinality: 8.0
Confidence:  112.50%
Recovered:   8 tokens
FULL ROUNDTRIP -- all 8 unique tokens recovered


---
## 5. Batch Comparison Matrix
Pairwise BSS inclusion matrix.

In [6]:
docs = {
    "ML": "machine learning deep learning neural networks supervised unsupervised",
    "Physics": "quantum mechanics relativity particles waves subatomic nuclear",
    "Biology": "dna rna protein cell membrane mitochondria evolution genome",
    "ML+Phys": "quantum machine learning tensor networks neural quantum states",
}

print(f'{"":12}', end="")
for name in docs:
    print(f'{name:>12}', end="")
print()

for name_a, text_a in docs.items():
    print(f'{name_a:12}', end="")
    for name_b, text_b in docs.items():
        tau = bss_inclusion(text_a, text_b)
        print(f'{tau:>12.3f}', end="")
    print()

print()
print("Rows = BSS tau(row -> column). Higher = more similar.")


                      ML     Physics     Biology     ML+Phys
ML                 1.000       0.000       0.000       0.571
Physics            0.000       1.000       0.000       0.143
Biology            0.000       0.000       1.000       0.000
ML+Phys            0.571       0.143       0.000       1.000

Rows = BSS tau(row -> column). Higher = more similar.


---
## 6. IPFS Storage
Content-addressed HLLSet persistence.

In [7]:
import subprocess as sp

def ipfs_running():
    try:
        sp.run(["ipfs", "id"], capture_output=True, timeout=3)
        return True
    except:
        return False

if ipfs_running():
    print("IPFS daemon is running")
else:
    print("IPFS daemon not running (using in-memory store/load)")

# Store + load in ONE Lua call (each hllset() call = fresh DslRuntime)
lt = lua_list(['persistent', 'data', 'ipfs'])
result = hllset(
    "local e = hllset.inscribe(" + lt + "); "
    "hllset.store(e); "
    "local loaded = hllset.load(e:key()); "
    "return {key=e:key(), exists=hllset.exists(e:key()), card=#loaded}"
)

k = result.get('key', 'n/a')
x = result.get('exists', False)
c = result.get('card', 0)
print(f"Key:          {k}")
print(f"Exists:       {x}")
print(f"Loaded card:  {c}")
print("Store/load roundtrip OK")


IPFS daemon is running
Key:          h:80e886511d1c82ff47409bcca174ca4a37419b1f
Exists:       True
Loaded card:  3.0
Store/load roundtrip OK


---
## Summary

| Op | Mechanism |
|---|---|
| Tokenize | Text -> 1024-bit fingerprint |
| BSS tau | \|A n B\| / \|B\| |
| Jaccard | \|A n B\| / \|A u B\| |
| A+B | CRDT merge (bitwise OR) |
| A*B | Shared content (bitwise AND) |
| Materialize | HLLSet -> tokens via TokenLUT |
| Store/Load | HLLSet <-> IPFS |

All O(1). Batch matrix < 1 second.